# LLM_GPU Notebook

Converted from `LLM_GPU.py` for notebook-first workflow.

## 1) Core Setup + Data Pipeline

In [ ]:
import argparse
import json
import math
import os
import random
import sys
from collections import Counter
from dataclasses import asdict, dataclass
from typing import Dict, List, Optional, Sequence, Tuple

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler


CONTEXT_START_TOKEN = "<CS>"
CONTEXT_END_TOKEN = "<CE>"
RESPONSE_START_TOKEN = "<RS>"
RESPONSE_END_TOKEN = "<RE>"
UNKNOWN_TOKEN = "<UNK>"
PAD_TOKEN = "<PAD>"


@dataclass
class DataConfig:
    messages_path: str = "Messages.txt"
    max_context_characters: int = 1600
    max_response_characters: int = 400
    max_sequence_tokens: int = 2048
    max_samples: int = 0  # 0 means all
    max_vocab_size: int = 32768
    history_turns: int = 16
    history_separator: str = "\n"
    user_role_prefix: str = "User: "
    bot_role_prefix: str = "Bot: "
    use_global_history_fallback: bool = False


@dataclass
class ModelConfig:
    n_layer: int = 20
    n_head: int = 16
    n_embd: int = 1024
    block_size: int = 2048
    dropout: float = 0.0
    tie_embeddings: bool = True


@dataclass
class TrainConfig:
    seed: int = 42
    max_steps: int = 15000
    batch_size: int = 4
    grad_accum_steps: int = 16
    learning_rate: float = 2e-4
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    warmup_steps: int = 500
    min_lr: float = 3e-5
    grad_clip: float = 1.0
    log_interval: int = 20
    eval_interval: int = 200
    eval_batches: int = 20
    num_workers: int = 2
    out_dir: str = "checkpoints_gpu"
    compile_model: bool = False


@dataclass
class SpecialTokenIds:
    pad_id: int
    unk_id: int
    context_start_id: int
    context_end_id: int
    response_start_id: int
    response_end_id: int


SPECIAL_TOKENS = [
    PAD_TOKEN,
    UNKNOWN_TOKEN,
    CONTEXT_START_TOKEN,
    CONTEXT_END_TOKEN,
    RESPONSE_START_TOKEN,
    RESPONSE_END_TOKEN,
]

INVALID_DIALOG_ID_MARKERS = {
    "none",
    "null",
    "nan",
    "nil",
    "n/a",
    "na",
    "undefined",
}


def parse_dialog_id_suffix(raw_suffix: str) -> Optional[str]:
    stripped_suffix = raw_suffix.strip()
    if not stripped_suffix:
        return None
    if not (stripped_suffix.startswith("[") and stripped_suffix.endswith("]")):
        return None

    dialog_id = stripped_suffix[1:-1].strip()
    if not dialog_id:
        return None

    normalized_dialog_id = dialog_id.strip().strip("'\"").strip()
    if not normalized_dialog_id:
        return None

    if normalized_dialog_id.lower() in INVALID_DIALOG_ID_MARKERS:
        return None

    return normalized_dialog_id


def split_text_into_tokens(text: str) -> List[str]:
    tokens: List[str] = []
    current_word = ""

    for character in text:
        if character.isalnum() or character == "_":
            current_word += character
            continue

        if current_word:
            tokens.append(current_word)
            current_word = ""

        tokens.append(character)

    if current_word:
        tokens.append(current_word)

    return tokens


def parse_message_line(
    raw_line: str,
    max_context_characters: int,
    max_response_characters: int,
) -> Optional[Tuple[List[Tuple[str, str]], Optional[str]]]:
    stripped_line = raw_line.strip()
    if not stripped_line:
        return None

    extracted_turns: List[Tuple[str, str]] = []
    cursor = 0

    while True:
        context_start_index = stripped_line.find(CONTEXT_START_TOKEN, cursor)
        if context_start_index == -1:
            break

        context_text_start = context_start_index + len(CONTEXT_START_TOKEN)
        context_end_index = stripped_line.find(CONTEXT_END_TOKEN, context_text_start)
        if context_end_index == -1:
            return None

        response_start_index = stripped_line.find(
            RESPONSE_START_TOKEN, context_end_index + len(CONTEXT_END_TOKEN)
        )
        if response_start_index == -1:
            return None

        response_text_start = response_start_index + len(RESPONSE_START_TOKEN)
        response_end_index = stripped_line.find(RESPONSE_END_TOKEN, response_text_start)
        if response_end_index == -1:
            return None

        context_text = stripped_line[context_text_start:context_end_index]
        response_text = stripped_line[response_text_start:response_end_index]
        context_text = context_text.strip()[:max_context_characters]
        response_text = response_text.strip()[:max_response_characters]

        if context_text and response_text:
            extracted_turns.append((context_text, response_text))

        cursor = response_end_index + len(RESPONSE_END_TOKEN)

    if not extracted_turns:
        return None

    suffix_text = stripped_line[cursor:]
    dialog_id = parse_dialog_id_suffix(suffix_text)
    return extracted_turns, dialog_id


def build_sample_tokens(
    context_text: str,
    response_text: str,
    max_sequence_tokens: int,
) -> Tuple[List[str], int]:
    context_tokens = split_text_into_tokens(context_text)
    response_tokens = split_text_into_tokens(response_text)

    max_response_tokens = max_sequence_tokens - 4
    if len(response_tokens) > max_response_tokens:
        response_tokens = response_tokens[:max_response_tokens]

    max_context_tokens = max_sequence_tokens - (len(response_tokens) + 4)
    if max_context_tokens < 0:
        max_context_tokens = 0

    if len(context_tokens) > max_context_tokens:
        context_tokens = context_tokens[-max_context_tokens:]

    prefix_tokens = [
        CONTEXT_START_TOKEN,
        *context_tokens,
        CONTEXT_END_TOKEN,
        RESPONSE_START_TOKEN,
    ]

    full_tokens = [*prefix_tokens, *response_tokens, RESPONSE_END_TOKEN]
    response_target_start_index = len(prefix_tokens)

    return full_tokens, response_target_start_index


def build_history_context(
    records: Sequence[Tuple[str, str, Optional[str]]],
    sample_index: int,
    history_turns: int,
    history_separator: str,
    user_role_prefix: str,
    bot_role_prefix: str,
) -> str:
    clamped_history_turns = max(1, history_turns)
    current_dialog_id = records[sample_index][2]

    context_parts: List[str] = []
    if current_dialog_id:
        prior_records: List[Tuple[str, str]] = []
        for turn_index in range(sample_index - 1, -1, -1):
            previous_user_text, previous_bot_text, previous_dialog_id = records[turn_index]
            if previous_dialog_id != current_dialog_id:
                continue
            prior_records.append((previous_user_text, previous_bot_text))
            if len(prior_records) >= clamped_history_turns - 1:
                break
        prior_records.reverse()
    else:
        start_index = max(0, sample_index - (clamped_history_turns - 1))
        prior_records = []
        for turn_index in range(start_index, sample_index):
            previous_user_text, previous_bot_text, _previous_dialog_id = records[turn_index]
            prior_records.append((previous_user_text, previous_bot_text))

    for previous_user_text, previous_bot_text in prior_records:
        context_parts.append(f"{user_role_prefix}{previous_user_text}")
        context_parts.append(f"{bot_role_prefix}{previous_bot_text}")

    current_user_text, _current_response_text, _current_dialog_id = records[sample_index]
    context_parts.append(f"{user_role_prefix}{current_user_text}")

    return history_separator.join(context_parts)


def build_inference_context(
    history_pairs: Sequence[Tuple[str, str]],
    current_user_text: str,
    history_turns: int,
    history_separator: str,
    user_role_prefix: str,
    bot_role_prefix: str,
) -> str:
    clamped_history_turns = max(1, history_turns)
    prior_turns_to_keep = max(0, clamped_history_turns - 1)
    start_index = max(0, len(history_pairs) - prior_turns_to_keep)

    context_parts: List[str] = []
    for previous_user_text, previous_bot_text in history_pairs[start_index:]:
        context_parts.append(f"{user_role_prefix}{previous_user_text}")
        context_parts.append(f"{bot_role_prefix}{previous_bot_text}")

    context_parts.append(f"{user_role_prefix}{current_user_text}")
    return history_separator.join(context_parts)


def load_samples(data_cfg: DataConfig) -> List[Tuple[List[str], int]]:
    if not os.path.exists(data_cfg.messages_path):
        raise FileNotFoundError(
            f"Messages file was not found: {data_cfg.messages_path}. "
            "Create it in <CS>...<CE> <RS>...<RE> format "
            "(single-turn or multi-turn blocks per line)."
        )

    raw_records: List[Tuple[str, str, Optional[str]]] = []
    total_lines = 0
    valid_lines = 0
    skipped_lines = 0
    expanded_turns_from_multiturn_lines = 0
    auto_generated_dialog_ids = 0
    auto_generated_singleturn_dialog_ids = 0
    auto_generated_multiturn_dialog_ids = 0
    explicit_dialog_id_pairs = 0

    with open(data_cfg.messages_path, "r", encoding="utf-8") as source_file:
        for raw_line in source_file:
            total_lines += 1

            parsed = parse_message_line(
                raw_line,
                max_context_characters=data_cfg.max_context_characters,
                max_response_characters=data_cfg.max_response_characters,
            )
            if parsed is None:
                skipped_lines += 1
                continue

            turns, parsed_dialog_id = parsed
            valid_lines += 1
            if len(turns) > 1:
                expanded_turns_from_multiturn_lines += len(turns)

            effective_dialog_id = parsed_dialog_id
            if effective_dialog_id is None and len(turns) > 1:
                # Keep all turns from one multi-turn line in the same dialog history.
                effective_dialog_id = f"auto_line_{total_lines}"
                auto_generated_dialog_ids += 1
                auto_generated_multiturn_dialog_ids += 1
            elif effective_dialog_id is None and not data_cfg.use_global_history_fallback:
                # By default, avoid cross-line history mixing when dialog_id is missing.
                effective_dialog_id = f"auto_line_{total_lines}"
                auto_generated_dialog_ids += 1
                auto_generated_singleturn_dialog_ids += 1

            for context_text, response_text in turns:
                raw_records.append((context_text, response_text, effective_dialog_id))
                if parsed_dialog_id is not None:
                    explicit_dialog_id_pairs += 1
                if data_cfg.max_samples > 0 and len(raw_records) >= data_cfg.max_samples:
                    break

            if data_cfg.max_samples > 0 and len(raw_records) >= data_cfg.max_samples:
                break

    samples: List[Tuple[List[str], int]] = []
    for sample_index, (_context_text, response_text, _dialog_id) in enumerate(raw_records):
        expanded_context_text = build_history_context(
            raw_records,
            sample_index=sample_index,
            history_turns=data_cfg.history_turns,
            history_separator=data_cfg.history_separator,
            user_role_prefix=data_cfg.user_role_prefix,
            bot_role_prefix=data_cfg.bot_role_prefix,
        )
        sample_tokens, response_start = build_sample_tokens(
            expanded_context_text,
            response_text,
            max_sequence_tokens=data_cfg.max_sequence_tokens,
        )
        samples.append((sample_tokens, response_start))

    if not samples:
        raise RuntimeError(
            "No valid training samples were loaded. Expected format: "
            f"{CONTEXT_START_TOKEN}<context>{CONTEXT_END_TOKEN} "
            f"{RESPONSE_START_TOKEN}<response>{RESPONSE_END_TOKEN} "
            "(single-turn or repeated multi-turn blocks in one line)."
        )

    print(f"Loaded lines: {total_lines}")
    print(f"Valid lines: {valid_lines}")
    print(f"Valid pairs: {len(raw_records)}")
    print(f"Built samples: {len(samples)}")
    print(f"Skipped lines: {skipped_lines}")
    print(
        "Expanded turns from multi-turn lines: "
        f"{expanded_turns_from_multiturn_lines}"
    )
    pairs_with_any_dialog_id = sum(1 for _context, _response, dialog_id in raw_records if dialog_id)
    print(f"Pairs with [dialog_id] suffix: {explicit_dialog_id_pairs}")
    print(f"Pairs with effective dialog id: {pairs_with_any_dialog_id}")
    if auto_generated_dialog_ids > 0:
        print(
            "Auto-generated dialog ids (all): "
            f"{auto_generated_dialog_ids}"
        )
        print(
            "Auto-generated single-turn dialog ids: "
            f"{auto_generated_singleturn_dialog_ids}"
        )
        print(
            "Auto-generated multi-turn dialog ids: "
            f"{auto_generated_multiturn_dialog_ids}"
        )
    if explicit_dialog_id_pairs == 0:
        if data_cfg.use_global_history_fallback:
            print(
                "Dialog-aware history fallback: no valid [dialog_id] tags found, "
                "using global history."
            )
        else:
            print(
                "Dialog-aware history fallback: disabled, using auto ids for lines "
                "without valid [dialog_id]."
            )
    elif not data_cfg.use_global_history_fallback:
        print("Global history fallback: disabled for lines without valid [dialog_id].")
    print(f"History turns per sample: {max(1, data_cfg.history_turns)}")
    return samples


def build_vocabulary(
    samples: Sequence[Tuple[List[str], int]],
    max_vocab_size: int,
) -> Tuple[Dict[str, int], List[str], SpecialTokenIds]:
    token_counter: Counter = Counter()

    for token_list, _ in samples:
        token_counter.update(token_list)

    sorted_by_freq_then_token = sorted(
        token_counter.items(), key=lambda item: (-item[1], item[0])
    )

    kept_tokens = [token for token, _ in sorted_by_freq_then_token if token not in SPECIAL_TOKENS]
    max_non_special = max(0, max_vocab_size - len(SPECIAL_TOKENS))
    kept_tokens = kept_tokens[:max_non_special]

    id_to_token = [*SPECIAL_TOKENS, *kept_tokens]
    token_to_id = {token: token_id for token_id, token in enumerate(id_to_token)}

    special_token_ids = SpecialTokenIds(
        pad_id=token_to_id[PAD_TOKEN],
        unk_id=token_to_id[UNKNOWN_TOKEN],
        context_start_id=token_to_id[CONTEXT_START_TOKEN],
        context_end_id=token_to_id[CONTEXT_END_TOKEN],
        response_start_id=token_to_id[RESPONSE_START_TOKEN],
        response_end_id=token_to_id[RESPONSE_END_TOKEN],
    )

    kept_fraction = 100.0
    if token_counter:
        kept_fraction = 100.0 * sum(token_counter[token] for token in kept_tokens) / sum(
            token_counter.values()
        )

    print(f"Vocabulary size: {len(id_to_token)} (cap={max_vocab_size})")
    print(f"Token coverage after cap: {kept_fraction:.2f}%")

    return token_to_id, id_to_token, special_token_ids


def encode_tokens(token_list: Sequence[str], token_to_id: Dict[str, int], unk_id: int) -> List[int]:
    return [token_to_id.get(token, unk_id) for token in token_list]


class MessageDataset(Dataset):
    def __init__(
        self,
        samples: Sequence[Tuple[List[str], int]],
        token_to_id: Dict[str, int],
        special_ids: SpecialTokenIds,
        block_size: int,
    ):
        self.inputs: List[List[int]] = []
        self.labels: List[List[int]] = []
        self.pad_id = special_ids.pad_id

        for tokens, response_start in samples:
            token_ids = encode_tokens(tokens, token_to_id, special_ids.unk_id)

            if len(token_ids) < 2:
                continue

            max_token_length = block_size + 1
            if len(token_ids) > max_token_length:
                overflow = len(token_ids) - max_token_length
                token_ids = token_ids[overflow:]
                response_start = max(0, response_start - overflow)

            input_ids = token_ids[:-1]
            label_ids = token_ids[1:]

            ignore_until = max(0, response_start - 1)
            ignore_until = min(ignore_until, len(label_ids))
            for index in range(ignore_until):
                label_ids[index] = -100

            self.inputs.append(input_ids)
            self.labels.append(label_ids)

        if not self.inputs:
            raise RuntimeError("No usable sequences were built for training.")

    def __len__(self) -> int:
        return len(self.inputs)

    def __getitem__(self, index: int) -> Tuple[List[int], List[int]]:
        return self.inputs[index], self.labels[index]


def collate_batch(
    batch: Sequence[Tuple[List[int], List[int]]],
    pad_id: int,
) -> Tuple[torch.Tensor, torch.Tensor]:
    batch_size = len(batch)
    max_len = max(len(item[0]) for item in batch)

    input_tensor = torch.full((batch_size, max_len), pad_id, dtype=torch.long)
    label_tensor = torch.full((batch_size, max_len), -100, dtype=torch.long)

    for row_index, (input_ids, label_ids) in enumerate(batch):
        length = len(input_ids)
        input_tensor[row_index, :length] = torch.tensor(input_ids, dtype=torch.long)
        label_tensor[row_index, :length] = torch.tensor(label_ids, dtype=torch.long)

    return input_tensor, label_tensor




## 2) Model + Training Utilities

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        if cfg.n_embd % cfg.n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")

        self.n_head = cfg.n_head
        self.head_dim = cfg.n_embd // cfg.n_head
        self.dropout = nn.Dropout(cfg.dropout)

        self.qkv_proj = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=True)
        self.out_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, channels = x.shape

        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(channels, dim=2)

        q = q.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)

        dropout_probability = self.dropout.p if self.training else 0.0
        attention_output = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=dropout_probability,
            is_causal=True,
        )

        attention_output = attention_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len, channels
        )

        output = self.out_proj(attention_output)
        return self.dropout(output)


class MLP(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        inner_dim = 4 * cfg.n_embd
        self.fc1 = nn.Linear(cfg.n_embd, inner_dim, bias=True)
        self.fc2 = nn.Linear(inner_dim, cfg.n_embd, bias=True)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        return self.dropout(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg: ModelConfig, vocab_size: int):
        super().__init__()
        self.cfg = cfg
        self.vocab_size = vocab_size

        self.token_embedding = nn.Embedding(vocab_size, cfg.n_embd)
        self.position_embedding = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layer)])
        self.final_norm = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, vocab_size, bias=False)

        if cfg.tie_embeddings:
            self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self,
        input_ids: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        batch_size, seq_len = input_ids.shape
        if seq_len > self.cfg.block_size:
            raise ValueError(
                f"Input sequence length {seq_len} exceeds block size {self.cfg.block_size}."
            )

        positions = torch.arange(0, seq_len, device=input_ids.device)

        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None, :, :]
        x = self.dropout(x)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                labels.view(-1),
                ignore_index=-100,
            )

        return logits, loss


class CosineWarmupScheduler:
    def __init__(self, optimizer: torch.optim.Optimizer, cfg: TrainConfig):
        self.optimizer = optimizer
        self.cfg = cfg

    def step(self, current_step: int) -> float:
        if current_step < self.cfg.warmup_steps:
            lr = self.cfg.learning_rate * (current_step + 1) / max(1, self.cfg.warmup_steps)
        else:
            progress = (current_step - self.cfg.warmup_steps) / max(
                1, self.cfg.max_steps - self.cfg.warmup_steps
            )
            cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
            lr = self.cfg.min_lr + cosine * (self.cfg.learning_rate - self.cfg.min_lr)

        for param_group in self.optimizer.param_groups:
            param_group["lr"] = lr
        return lr


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())


def distributed_setup() -> Tuple[bool, int, int, int, torch.device]:
    rank = int(os.environ.get("RANK", "0"))
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    is_distributed = world_size > 1

    if is_distributed:
        if not torch.cuda.is_available():
            raise RuntimeError("DDP was requested but CUDA is not available.")
        torch.cuda.set_device(local_rank)
        dist.init_process_group(backend="nccl")
        device = torch.device("cuda", local_rank)
    else:
        if torch.cuda.is_available():
            device = torch.device("cuda")
        else:
            device = torch.device("cpu")

    return is_distributed, rank, local_rank, world_size, device


def distributed_cleanup(is_distributed: bool) -> None:
    if is_distributed and dist.is_initialized():
        dist.destroy_process_group()


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def evaluate(
    model: GPTModel,
    data_loader: DataLoader,
    device: torch.device,
    eval_batches: int,
    use_autocast: bool,
    amp_dtype: torch.dtype,
) -> float:
    model.eval()
    losses = []

    with torch.no_grad():
        for batch_index, (input_ids, labels) in enumerate(data_loader):
            if batch_index >= eval_batches:
                break

            input_ids = input_ids.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_autocast):
                _, loss = model(input_ids, labels)

            if loss is not None:
                losses.append(loss.item())

    model.train()
    if not losses:
        return float("nan")
    return sum(losses) / len(losses)


def save_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    step: int,
    token_to_id: Dict[str, int],
    id_to_token: List[str],
    data_cfg: DataConfig,
    model_cfg: ModelConfig,
    train_cfg: TrainConfig,
) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    raw_model = model.module if isinstance(model, nn.parallel.DistributedDataParallel) else model

    checkpoint = {
        "model_state_dict": raw_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "step": step,
        "token_to_id": token_to_id,
        "id_to_token": id_to_token,
        "data_config": asdict(data_cfg),
        "model_config": asdict(model_cfg),
        "train_config": asdict(train_cfg),
    }
    torch.save(checkpoint, path)


@torch.no_grad()
def generate_response(
    model: GPTModel,
    context_text: str,
    token_to_id: Dict[str, int],
    id_to_token: List[str],
    special_ids: SpecialTokenIds,
    max_new_tokens: int = 200,
    temperature: float = 0.8,
    top_k: int = 40,
    greedy: bool = False,
) -> str:
    model.eval()
    device = next(model.parameters()).device

    context_tokens = split_text_into_tokens(context_text)
    prompt_tokens = [
        CONTEXT_START_TOKEN,
        *context_tokens,
        CONTEXT_END_TOKEN,
        RESPONSE_START_TOKEN,
    ]
    prompt_ids = encode_tokens(prompt_tokens, token_to_id, special_ids.unk_id)

    input_ids = torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, :]

    for _ in range(max_new_tokens):
        if input_ids.size(1) > model.cfg.block_size:
            input_cond = input_ids[:, -model.cfg.block_size :]
        else:
            input_cond = input_ids

        logits, _ = model(input_cond)
        next_logits = logits[:, -1, :]

        if greedy:
            next_id = torch.argmax(next_logits, dim=-1, keepdim=True)
        else:
            next_logits = next_logits / max(temperature, 1e-5)

            if top_k > 0:
                top_values, _ = torch.topk(next_logits, k=min(top_k, next_logits.size(-1)))
                cutoff = top_values[:, -1].unsqueeze(-1)
                next_logits = torch.where(
                    next_logits < cutoff,
                    torch.full_like(next_logits, float("-inf")),
                    next_logits,
                )

            probs = F.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        if next_id.item() == special_ids.response_end_id:
            break

        input_ids = torch.cat([input_ids, next_id], dim=1)

    generated_ids = input_ids[0, len(prompt_ids) :].tolist()
    generated_tokens = [id_to_token[token_id] for token_id in generated_ids if token_id < len(id_to_token)]
    return "".join(generated_tokens).strip()




## 3) Train / Chat Entrypoints

In [ ]:
def train(args: argparse.Namespace) -> None:
    data_cfg = DataConfig(
        messages_path=args.messages,
        max_context_characters=args.max_context_chars,
        max_response_characters=args.max_response_chars,
        max_sequence_tokens=args.max_sequence_tokens,
        max_samples=args.max_samples,
        max_vocab_size=args.max_vocab_size,
        history_turns=args.history_turns,
        history_separator=args.history_separator,
        user_role_prefix=args.user_role_prefix,
        bot_role_prefix=args.bot_role_prefix,
        use_global_history_fallback=args.use_global_history_fallback,
    )
    model_cfg = ModelConfig(
        n_layer=args.n_layer,
        n_head=args.n_head,
        n_embd=args.n_embd,
        block_size=args.block_size,
        dropout=args.dropout,
        tie_embeddings=not args.no_tie_embeddings,
    )
    train_cfg = TrainConfig(
        seed=args.seed,
        max_steps=args.max_steps,
        batch_size=args.batch_size,
        grad_accum_steps=args.grad_accum_steps,
        learning_rate=args.learning_rate,
        weight_decay=args.weight_decay,
        beta1=args.beta1,
        beta2=args.beta2,
        warmup_steps=args.warmup_steps,
        min_lr=args.min_lr,
        grad_clip=args.grad_clip,
        log_interval=args.log_interval,
        eval_interval=args.eval_interval,
        eval_batches=args.eval_batches,
        num_workers=args.num_workers,
        out_dir=args.out_dir,
        compile_model=args.compile_model,
    )

    is_distributed, rank, _local_rank, world_size, device = distributed_setup()
    is_main_process = rank == 0

    try:
        set_seed(train_cfg.seed + rank)

        samples = load_samples(data_cfg)
        token_to_id, id_to_token, special_ids = build_vocabulary(
            samples,
            max_vocab_size=data_cfg.max_vocab_size,
        )

        dataset = MessageDataset(
            samples,
            token_to_id=token_to_id,
            special_ids=special_ids,
            block_size=model_cfg.block_size,
        )

        split_index = int(0.98 * len(dataset))
        if split_index <= 0:
            split_index = len(dataset)
        if split_index >= len(dataset):
            split_index = len(dataset) - 1 if len(dataset) > 1 else len(dataset)
        train_indices = list(range(0, split_index))
        eval_indices = list(range(split_index, len(dataset)))
        train_subset = torch.utils.data.Subset(dataset, train_indices)
        eval_subset = torch.utils.data.Subset(dataset, eval_indices) if eval_indices else None

        if len(train_subset) < train_cfg.batch_size:
            raise RuntimeError(
                "Train split is smaller than batch_size. "
                "Lower --batch-size or provide more samples."
            )

        train_sampler = (
            DistributedSampler(train_subset, shuffle=True, drop_last=True)
            if is_distributed
            else None
        )

        train_loader = DataLoader(
            train_subset,
            batch_size=train_cfg.batch_size,
            shuffle=train_sampler is None,
            sampler=train_sampler,
            num_workers=train_cfg.num_workers,
            pin_memory=device.type == "cuda",
            drop_last=True,
            collate_fn=lambda batch: collate_batch(batch, special_ids.pad_id),
        )

        eval_loader = None
        if eval_subset is not None and len(eval_subset) > 0:
            eval_loader = DataLoader(
                eval_subset,
                batch_size=train_cfg.batch_size,
                shuffle=False,
                num_workers=max(1, train_cfg.num_workers // 2),
                pin_memory=device.type == "cuda",
                drop_last=False,
                collate_fn=lambda batch: collate_batch(batch, special_ids.pad_id),
            )

        model = GPTModel(model_cfg, vocab_size=len(id_to_token)).to(device)
        if train_cfg.compile_model:
            model = torch.compile(model)

        if is_distributed:
            model = nn.parallel.DistributedDataParallel(
                model,
                device_ids=[device.index],
                output_device=device.index,
                find_unused_parameters=False,
            )

        param_count = count_parameters(model)
        if is_main_process:
            print(f"Device: {device}")
            print(f"World size: {world_size}")
            print(f"Model parameters: {param_count:,}")
            print(
                "Target check (100M-ish): "
                f"{(param_count / 1_000_000):.2f}M parameters"
            )

        raw_model = model.module if isinstance(model, nn.parallel.DistributedDataParallel) else model

        optimizer_kwargs = dict(
            params=raw_model.parameters(),
            lr=train_cfg.learning_rate,
            betas=(train_cfg.beta1, train_cfg.beta2),
            weight_decay=train_cfg.weight_decay,
        )
        if device.type == "cuda":
            try:
                optimizer = torch.optim.AdamW(**optimizer_kwargs, fused=True)
            except TypeError:
                optimizer = torch.optim.AdamW(**optimizer_kwargs)
        else:
            optimizer = torch.optim.AdamW(**optimizer_kwargs)
        scheduler = CosineWarmupScheduler(optimizer, train_cfg)

        use_autocast = device.type == "cuda"
        amp_dtype = (
            torch.bfloat16
            if device.type == "cuda" and torch.cuda.is_bf16_supported()
            else torch.float16
        )

        if is_main_process:
            os.makedirs(train_cfg.out_dir, exist_ok=True)
            config_payload = {
                "data_config": asdict(data_cfg),
                "model_config": asdict(model_cfg),
                "train_config": asdict(train_cfg),
                "param_count": param_count,
                "vocab_size": len(id_to_token),
            }
            with open(
                os.path.join(train_cfg.out_dir, "run_config.json"),
                "w",
                encoding="utf-8",
            ) as config_file:
                json.dump(config_payload, config_file, ensure_ascii=False, indent=2)

        train_iterator = iter(train_loader)
        sampler_epoch = 0
        if is_distributed and train_sampler is not None:
            train_sampler.set_epoch(sampler_epoch)

        for step in range(train_cfg.max_steps):
            optimizer.zero_grad(set_to_none=True)
            accumulated_loss = 0.0

            for _ in range(train_cfg.grad_accum_steps):
                try:
                    input_ids, labels = next(train_iterator)
                except StopIteration:
                    sampler_epoch += 1
                    if is_distributed and train_sampler is not None:
                        train_sampler.set_epoch(sampler_epoch)
                    train_iterator = iter(train_loader)
                    input_ids, labels = next(train_iterator)

                input_ids = input_ids.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                with torch.autocast(
                    device_type=device.type,
                    dtype=amp_dtype,
                    enabled=use_autocast,
                ):
                    _, loss = model(input_ids, labels)
                    loss = loss / train_cfg.grad_accum_steps

                loss.backward()
                accumulated_loss += loss.item()

            if train_cfg.grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(raw_model.parameters(), train_cfg.grad_clip)

            current_lr = scheduler.step(step)
            optimizer.step()

            if is_main_process and (step % train_cfg.log_interval == 0 or step == train_cfg.max_steps - 1):
                print(
                    f"step {step + 1:5d}/{train_cfg.max_steps} "
                    f"| train_loss {accumulated_loss:.4f} "
                    f"| lr {current_lr:.6e}"
                )

            if (
                is_main_process
                and eval_loader is not None
                and train_cfg.eval_interval > 0
                and (step + 1) % train_cfg.eval_interval == 0
            ):
                eval_loss = evaluate(
                    raw_model,
                    eval_loader,
                    device,
                    eval_batches=train_cfg.eval_batches,
                    use_autocast=use_autocast,
                    amp_dtype=amp_dtype,
                )
                print(f"eval @ step {step + 1}: loss {eval_loss:.4f}")

                checkpoint_path = os.path.join(
                    train_cfg.out_dir,
                    f"checkpoint_step_{step + 1}.pt",
                )
                save_checkpoint(
                    checkpoint_path,
                    model,
                    optimizer,
                    step + 1,
                    token_to_id,
                    id_to_token,
                    data_cfg,
                    model_cfg,
                    train_cfg,
                )

        if is_main_process:
            final_path = os.path.join(train_cfg.out_dir, "checkpoint_final.pt")
            save_checkpoint(
                final_path,
                model,
                optimizer,
                train_cfg.max_steps,
                token_to_id,
                id_to_token,
                data_cfg,
                model_cfg,
                train_cfg,
            )
            preview = generate_response(
                raw_model,
                "Give me a short plan for today's learning session.",
                token_to_id,
                id_to_token,
                special_ids,
                max_new_tokens=80,
            )
            print("Sample generation:")
            print(preview)
    finally:
        distributed_cleanup(is_distributed)


def chat(args: argparse.Namespace) -> None:
    checkpoint = torch.load(args.checkpoint, map_location="cpu")

    model_cfg = ModelConfig(**checkpoint["model_config"])
    data_cfg_payload = checkpoint.get("data_config", {})
    data_cfg = DataConfig(**data_cfg_payload) if data_cfg_payload else DataConfig()
    id_to_token = checkpoint["id_to_token"]
    token_to_id = checkpoint["token_to_id"]

    special_ids = SpecialTokenIds(
        pad_id=token_to_id[PAD_TOKEN],
        unk_id=token_to_id[UNKNOWN_TOKEN],
        context_start_id=token_to_id[CONTEXT_START_TOKEN],
        context_end_id=token_to_id[CONTEXT_END_TOKEN],
        response_start_id=token_to_id[RESPONSE_START_TOKEN],
        response_end_id=token_to_id[RESPONSE_END_TOKEN],
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GPTModel(model_cfg, vocab_size=len(id_to_token)).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    mode_label = "greedy" if args.greedy else "sampling"
    print(f"Interactive chat mode ({mode_label}). Type 'exit' to quit.")
    conversation_pairs: List[Tuple[str, str]] = []

    while True:
        try:
            user_input = input("\n<context> ").strip()
        except EOFError:
            break

        if user_input.lower() in {"exit", "quit", "выход"}:
            break

        if not user_input:
            print("Context is empty.")
            continue

        model_context = build_inference_context(
            history_pairs=conversation_pairs,
            current_user_text=user_input,
            history_turns=data_cfg.history_turns,
            history_separator=data_cfg.history_separator,
            user_role_prefix=data_cfg.user_role_prefix,
            bot_role_prefix=data_cfg.bot_role_prefix,
        )
        response = generate_response(
            model,
            model_context,
            token_to_id,
            id_to_token,
            special_ids,
            max_new_tokens=args.max_new_tokens,
            temperature=args.temperature,
            top_k=args.top_k,
            greedy=args.greedy,
        )
        conversation_pairs.append((user_input, response))
        print(f"<response> {response}")


def build_argument_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="GPU-ready PyTorch LLM trainer for Messages.txt",
    )

    subparsers = parser.add_subparsers(dest="mode", required=False)

    train_parser = subparsers.add_parser("train", help="Train a larger GPT model")
    train_parser.add_argument("--messages", type=str, default="Messages.txt")
    train_parser.add_argument("--max-context-chars", type=int, default=1600)
    train_parser.add_argument("--max-response-chars", type=int, default=400)
    train_parser.add_argument("--max-sequence-tokens", type=int, default=2048)
    train_parser.add_argument("--max-samples", type=int, default=0)
    train_parser.add_argument("--max-vocab-size", type=int, default=32768)
    train_parser.add_argument("--history-turns", type=int, default=16)
    train_parser.add_argument("--history-separator", type=str, default="\n")
    train_parser.add_argument("--user-role-prefix", type=str, default="User: ")
    train_parser.add_argument("--bot-role-prefix", type=str, default="Bot: ")
    train_parser.add_argument(
        "--use-global-history-fallback",
        action="store_true",
        help=(
            "Enable old behavior: if [dialog_id] is missing/invalid, use recent "
            "global lines as history context."
        ),
    )

    train_parser.add_argument("--n-layer", type=int, default=20)
    train_parser.add_argument("--n-head", type=int, default=16)
    train_parser.add_argument("--n-embd", type=int, default=1024)
    train_parser.add_argument("--block-size", type=int, default=2048)
    train_parser.add_argument("--dropout", type=float, default=0.0)
    train_parser.add_argument("--no-tie-embeddings", action="store_true")

    train_parser.add_argument("--seed", type=int, default=42)
    train_parser.add_argument("--max-steps", type=int, default=15000)
    train_parser.add_argument("--batch-size", type=int, default=4)
    train_parser.add_argument("--grad-accum-steps", type=int, default=16)
    train_parser.add_argument("--learning-rate", type=float, default=2e-4)
    train_parser.add_argument("--weight-decay", type=float, default=0.1)
    train_parser.add_argument("--beta1", type=float, default=0.9)
    train_parser.add_argument("--beta2", type=float, default=0.95)
    train_parser.add_argument("--warmup-steps", type=int, default=500)
    train_parser.add_argument("--min-lr", type=float, default=3e-5)
    train_parser.add_argument("--grad-clip", type=float, default=1.0)
    train_parser.add_argument("--log-interval", type=int, default=20)
    train_parser.add_argument("--eval-interval", type=int, default=200)
    train_parser.add_argument("--eval-batches", type=int, default=20)
    train_parser.add_argument("--num-workers", type=int, default=2)
    train_parser.add_argument("--out-dir", type=str, default="checkpoints_gpu")
    train_parser.add_argument("--compile-model", action="store_true")

    chat_parser = subparsers.add_parser("chat", help="Run interactive chat from checkpoint")
    chat_parser.add_argument("--checkpoint", type=str, required=True)
    chat_parser.add_argument("--max-new-tokens", type=int, default=160)
    chat_parser.add_argument("--temperature", type=float, default=0.8)
    chat_parser.add_argument("--top-k", type=int, default=40)
    chat_parser.add_argument(
        "--greedy",
        action="store_true",
        help="Use deterministic argmax decoding (ignores temperature/top-k).",
    )

    return parser


def _is_running_in_notebook() -> bool:
    if "ipykernel" in sys.modules:
        return True
    return False


def _parse_args_notebook_safe(
    parser: argparse.ArgumentParser,
    argv: Optional[Sequence[str]] = None,
) -> Optional[argparse.Namespace]:
    if argv is not None:
        return parser.parse_args(list(argv))

    args, unknown = parser.parse_known_args()

    if args.mode is None:
        if _is_running_in_notebook():
            parser.print_help()
            print(
                "\nNotebook hint:\n"
                "Call main([...]) with explicit args, for example:\n"
                "main([\"train\", \"--messages\", \"/content/Messages.txt\"])"
            )
            return None
        parser.error("Missing mode argument. Choose one of: train, chat")

    if unknown:
        print(f"Warning: ignoring unknown CLI args: {unknown}")

    return args


def main(argv: Optional[Sequence[str]] = None) -> None:
    parser = build_argument_parser()
    args = _parse_args_notebook_safe(parser, argv=argv)
    if args is None:
        return

    if args.mode == "train":
        train(args)
    elif args.mode == "chat":
        chat(args)
    else:
        raise ValueError(f"Unknown mode: {args.mode}")


if __name__ == "__main__" and not _is_running_in_notebook():
    main()


## 4) Drive + Local Runtime Setup (plaintext stays local)


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = False
DRIVE_OK = False

try:
    from google.colab import drive, files
    IN_COLAB = True
except Exception:
    drive = None
    files = None

if IN_COLAB:
    try:
        drive.mount('/content/drive', force_remount=True)
        DRIVE_ROOT = '/content/drive/MyDrive/LLM_GPU'
        DRIVE_OK = True
    except Exception as error:
        print('Drive mount failed:', error)
        DRIVE_ROOT = '/content/LLM_GPU_drive_fallback'
else:
    DRIVE_ROOT = str(Path.cwd() / 'LLM_GPU')

if IN_COLAB:
    LOCAL_ROOT = '/content/LLM_GPU_runtime'
else:
    LOCAL_ROOT = str(Path.cwd() / 'LLM_GPU_runtime')

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(LOCAL_ROOT, exist_ok=True)

# Security policy:
# - keep encrypted file in Drive
# - keep decrypted plaintext ONLY in LOCAL_ROOT
MESSAGES_ENC_PATH = os.path.join(DRIVE_ROOT, 'Messages.txt.enc')
MESSAGES_DRIVE_PLAINTEXT_PATH = os.path.join(DRIVE_ROOT, 'Messages.txt')
MESSAGES_PATH = os.path.join(LOCAL_ROOT, 'Messages.txt')
OUT_DIR = os.path.join(DRIVE_ROOT, 'checkpoints_gpu_100m')


def sync_dir(src_dir: str, dst_dir: str) -> None:
    """Copy all files from src_dir to dst_dir using pure Python (no shell)."""
    src = Path(src_dir)
    dst = Path(dst_dir)
    if not src.exists():
        raise FileNotFoundError(f'Source directory does not exist: {src}')

    dst.mkdir(parents=True, exist_ok=True)

    for item in src.rglob('*'):
        rel = item.relative_to(src)
        target = dst / rel
        if item.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, target)

    print(f'Sync complete: {src} -> {dst}')


def decrypt_messages_enc_to_local(
    private_key_path: str,
    crypt_script_path: str = 'Messages crypt.py',
    passphrase_env: str = 'MSG_PRIVATE_KEY_PASSPHRASE',
) -> None:
    """Decrypt Drive .enc file to LOCAL plaintext path only."""
    if not os.path.exists(crypt_script_path):
        raise FileNotFoundError(f'Crypt script not found: {crypt_script_path}')
    if not os.path.exists(MESSAGES_ENC_PATH):
        raise FileNotFoundError(f'Encrypted file not found: {MESSAGES_ENC_PATH}')

    cmd = [
        sys.executable,
        crypt_script_path,
        'decrypt',
        '--input', MESSAGES_ENC_PATH,
        '--private-key', private_key_path,
        '--output', MESSAGES_PATH,
        '--overwrite',
        '--private-passphrase-env', passphrase_env,
    ]
    subprocess.run(cmd, check=True)
    print('Decrypted to LOCAL path:', MESSAGES_PATH)


def cleanup_local_messages() -> None:
    if os.path.exists(MESSAGES_PATH):
        os.remove(MESSAGES_PATH)
        print('Removed local plaintext:', MESSAGES_PATH)
    else:
        print('Local plaintext not found:', MESSAGES_PATH)


print('IN_COLAB:', IN_COLAB)
print('DRIVE_OK:', DRIVE_OK)
print('DRIVE_ROOT:', DRIVE_ROOT)
print('LOCAL_ROOT:', LOCAL_ROOT)
print('MESSAGES_ENC_PATH (Drive):', MESSAGES_ENC_PATH)
print('MESSAGES_PATH (LOCAL plaintext):', MESSAGES_PATH)
print('OUT_DIR (Drive checkpoints):', OUT_DIR)

if DRIVE_OK and os.path.exists(MESSAGES_DRIVE_PLAINTEXT_PATH):
    print('[warn] Plaintext Messages.txt exists on Drive:', MESSAGES_DRIVE_PLAINTEXT_PATH)
    print('[warn] Training now uses LOCAL plaintext path only. Remove Drive plaintext manually for security.')

if IN_COLAB and (not DRIVE_OK):
    if os.path.exists('/content/Messages.txt') and not os.path.exists(MESSAGES_PATH):
        shutil.copy2('/content/Messages.txt', MESSAGES_PATH)
        print('Copied /content/Messages.txt ->', MESSAGES_PATH)
    elif (files is not None) and (not os.path.exists(MESSAGES_PATH)):
        print('Upload Messages.txt manually (fallback mode) ...')
        uploaded = files.upload()
        if 'Messages.txt' in uploaded:
            with open(MESSAGES_PATH, 'wb') as target_file:
                target_file.write(uploaded['Messages.txt'])
            print('Saved uploaded file to', MESSAGES_PATH)


Mounted at /content/drive
IN_COLAB: True
DRIVE_OK: True
DRIVE_ROOT: /content/drive/MyDrive/LLM_GPU
LOCAL_ROOT: /content/LLM_GPU_runtime
MESSAGES_ENC_PATH (Drive): /content/drive/MyDrive/LLM_GPU/Messages.txt.enc
MESSAGES_PATH (LOCAL plaintext): /content/LLM_GPU_runtime/Messages.txt
OUT_DIR (Drive checkpoints): /content/drive/MyDrive/LLM_GPU/checkpoints_gpu_100m


## 5) Usage Examples


In [ ]:
# Example: larger-model training on Drive mix dataset
# Assumes you uploaded: /content/drive/MyDrive/LLM_GPU/Messages_mix.txt

MIX_MESSAGES_PATH = '/content/drive/MyDrive/LLM_GPU/Messages_mix.txt'

main([
    'train',
    '--messages', MIX_MESSAGES_PATH,
    '--out-dir', OUT_DIR,
    '--max-context-chars', '1600',
    '--history-turns', '16',
    '--max-sequence-tokens', '2048',
    '--block-size', '2048',
    '--n-layer', '20',
    '--n-head', '16',
    '--n-embd', '1024',
    '--batch-size', '4',
    '--grad-accum-steps', '16',
    '--learning-rate', '2e-4',
    '--warmup-steps', '500',
    '--max-steps', '15000',
    '--eval-interval', '5000',
])

# Optional: separate long-context run with lower LR
# LONG_MESSAGES_PATH = '/content/drive/MyDrive/LLM_GPU/Messages_longContext.txt'
# main([
#     'train',
#     '--messages', LONG_MESSAGES_PATH,
#     '--out-dir', OUT_DIR,
#     '--max-context-chars', '1600',
#     '--history-turns', '16',
#     '--max-sequence-tokens', '2048',
#     '--block-size', '2048',
#     '--n-layer', '20',
#     '--n-head', '16',
#     '--n-embd', '1024',
#     '--batch-size', '4',
#     '--grad-accum-steps', '16',
#     '--learning-rate', '8e-5',
#     '--warmup-steps', '300',
#     '--max-steps', '3000',
#     '--eval-interval', '1000',
# ])


In [ ]:
# Example: interactive chat from final checkpoint on Google Drive (greedy mode)

main([
    'chat',
    '--checkpoint', os.path.join(OUT_DIR, 'checkpoint_final.pt'),
    '--greedy',
])


Interactive chat mode (greedy). Type 'exit' to quit.

<context> Кто ты такой?
<response> Я являюсь искусственным интеллектом, <UNK> компанией OpenAI. Моя основная задача — предоставлять информацию и помогать пользователям, отвечая на их вопросы и выполняя различные задачи. Если у вас есть вопросы или нужна помощь, не стесняйтесь обращаться!

<context> привет
<response> Я являюсь искусственным интеллектом, <<UNK>> компанией OpenAI. Моя основная задача — предоставлять информацию и помогать пользователям, отвечая на их вопросы и выполняя различные задачи. Если у вас есть вопросы или нужна помощь, не стесняйтесь обращаться!

<context> кдлоалуколпащуклощпалукщзп


In [ ]:
# Optional: if you trained to local /content, sync checkpoints to Drive after training

LOCAL_OUT_DIR = '/content/checkpoints_gpu_100m'
DRIVE_OUT_DIR = os.path.join(DRIVE_ROOT, 'checkpoints_gpu_100m')

if os.path.exists(LOCAL_OUT_DIR):
    sync_dir(LOCAL_OUT_DIR, DRIVE_OUT_DIR)
else:
    print(f'Local directory not found: {LOCAL_OUT_DIR}')


In [ ]:
# Optional security cleanup: remove local plaintext after training
cleanup_local_messages()


In [ ]:
# Multi-GPU launch (run in terminal, not inside notebook):
# torchrun --nproc_per_node=4 LLM_GPU.py train --messages Messages.txt --out-dir checkpoints_gpu_100m
